## Bronze Layer (Deliverable 1.1.1 and 1.1.2)

This notebook completes the Bronze layer of the Medallion Architecture for the pump sensor dataset. It focuses on raw data ingestion, structural validation, and standardization so that the rest of the analytical workflow has a clean and consistent foundation.

The key goals of this notebook are:

- To ingest the raw pump sensor data from the source files.
- To validate the structure, column consistency, data types, and timestamp integrity.
- To standardize the dataset into the unified Bronze schema, including meta fields, identity fields, and provenance tracking.
- To export the Bronze dataset in a reproducible format that can be used by the Silver Pre-EDA and Silver EDA notebooks.

Outputs from this notebook support the project write-up in Section B and Section C by:

- Establishing the standardized dataset required for all downstream analysis and modeling described in Section C.2 and C.2.A.
- Providing the meta-data, event identifiers, timestamp fields, and structural consistency checks required for later segmentation of normal vs abnormal periods used in Section C.4.
- Ensuring that the Silver and Gold layers receive a clean, well-formed dataset, enabling reliable feature behavior analysis, anomaly profiling, and model comparison.
- Serving as the reproducible starting point for the entire analytical pipeline, consistent with the Medallion Architecture described in B.3.

In [ ]:
print("hello")

## Environment Setup and Utility Imports

In this section I am loading the libraries and utility functions that support the Bronze stage of the pipeline.

The main goal here is to get everything ready before I touch the raw dataset. That includes general Python libraries, project path handling, ingestion helpers, logging, truth-record utilities, configuration loading, Weights & Biases support, and the PostgreSQL writing utilities I may use later in the workflow.

At this point, I am not transforming the data yet. I am just making sure the notebook has access to the tools it needs so the rest of the process stays organized and repeatable.

In [ ]:
import os
import glob
from pathlib import Path
import yaml
import sys

import json
import logging
import wandb
from datetime import datetime, timezone

from typing import Optional, Tuple, List

import pandas as pd 
import numpy as np

import hashlib

# Custom Utilities Module
from utils.paths import get_paths
from utils.file_io import ingest_data, save_data, load_json
from utils.logging_setup import configure_logging, log_layer_paths
from utils.eda_logging import profile_dataframe

from utils.truths import (
    make_process_run_id,
    build_file_fingerprint,
    extract_truth_hash,
    identify_meta_columns,
    identify_feature_columns,
    initialize_layer_truth,
    update_truth_section,
    build_truth_record,
    save_truth_record,
    append_truth_index,
    stamp_truth_columns,
    load_truth_record,
    find_truth_record_by_hash, 
    load_truth_record_by_hash, 
    load_parent_truth_record_from_dataframe,
)

'''
from utils.wandb_utils import (
    log_metrics,
    log_dataframe_head,
    log_dir_as_artifact,
    log_files_as_artifact,
)
'''
from utils.wandb_utils import finalize_wandb_stage

from utils.pipeline_config_loader import (
    load_pipeline_config,
    build_truth_config_block,
    set_wandb_dir_from_config,
    export_config_snapshot,
)

from utils.postgres_util import get_engine_from_env
from utils.layer_postgres_writer import read_layer_dataframe, write_layer_dataframe, prepare_layer_dataframe



# Show more columns
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)



----

## Load Project Paths, Configuration, and Runtime Settings

Here I am pulling in the resolved project paths and the Bronze configuration settings that control how this notebook runs.

This cell is doing a lot of the setup work for the pipeline:
- finding the correct folders
- loading the Bronze config
- resolving important file names
- defining dataset metadata
- setting up versioning values
- preparing output locations for Bronze data, artifacts, logs, and truths

I like doing this near the top because it keeps the rest of the notebook cleaner. Instead of hard-coding values all over the place, I can define them once here and reuse them throughout the Bronze workflow.

In [ ]:
# Get Path's Object
paths = get_paths()

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Get configs
#CONFIG_ROOT = Path("configs")
CONFIG_ROOT = paths.configs

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 


CONFIG_RUN_MODE = "train"
CONFIG_PROFILE = "default"


#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 


CONFIG = load_pipeline_config(
    config_root=CONFIG_ROOT,
    stage="bronze",
    dataset="pump",
    mode=CONFIG_RUN_MODE,
    profile=CONFIG_PROFILE,
    project_root=paths.root,
).data

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

BRONZE_CFG = CONFIG["bronze"]
PATHS = CONFIG["resolved_paths"]
FILENAMES = CONFIG["filenames"]
PIPELINE = CONFIG.get(
    "pipeline",
    {
        "execution_mode": "batch",
        "orchestration_mode": "notebook",
    },
)

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 


TRUTH_CONFIG = build_truth_config_block(CONFIG)
TRUTH_CONFIG["pipeline"] = PIPELINE

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 


# Paths
RAW_DATA_PATH = Path(PATHS["data_raw_dir"])
BRONZE_DATA_PATH = Path(PATHS["data_bronze_train_dir"])

BRONZE_ARTIFACTS_PATH = Path(PATHS["bronze_artifacts_dir"])
TRUTHS_PATH = Path(PATHS["truths_dir"])
TRUTH_INDEX_PATH = Path(PATHS["truth_index_path"])
LOGS_PATH = Path(PATHS["logs_root"])


#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# W&B
WANDB_DIR = set_wandb_dir_from_config(CONFIG)

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 


# Make sure paths exist
BRONZE_DATA_PATH.mkdir(parents=True, exist_ok=True)
BRONZE_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)
TRUTHS_PATH.mkdir(parents=True, exist_ok=True)
LOGS_PATH.mkdir(parents=True, exist_ok=True)

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 


# Meta details
LAYER_NAME = BRONZE_CFG["layer_name"]
BRONZE_VERSION = CONFIG["versions"]["bronze"]
TRUTH_VERSION = CONFIG["versions"]["truth"]

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Keep pipeline execution separate from run mode
PIPELINE_MODE = PIPELINE["execution_mode"]
RUN_MODE = CONFIG["runtime"]["mode"] 
CONFIG_PROFILE = CONFIG["runtime"]["profile"] 

# W&B
WANDB_PROJECT = CONFIG["wandb"]["project"]
WANDB_ENTITY = CONFIG["wandb"]["entity"]
WANDB_RUN_NAME = f"{BRONZE_VERSION}"

# Raw file details
RAW_FILE_PATH = Path(PATHS["raw_file_path"]).parent
RAW_FILE_NAME = CONFIG["dataset"]["raw_file_name"]

# Dataset details

DATASET_NAME_ARGUMENT = None
DATASET_NAME_CONFIG = CONFIG["dataset"]["name"]
DATASET_CANDIDATES = BRONZE_CFG["dataset_candidates"]

SPLIT_STATUS = CONFIG["dataset"]["split_status"]
LABEL_TYPE = CONFIG["dataset"]["label_type"]
LABEL_SOURCE = CONFIG["dataset"]["label_source"]
RUN_ID = CONFIG["dataset"]["run_id"]
ASSET_ID = CONFIG["dataset"]["asset_id"]

# Separate processing lineage id
PROCESS_RUN_ID = make_process_run_id(BRONZE_CFG["process_run_id_prefix"])

ADD_RECORD_ID = bool(BRONZE_CFG["add_record_id"])
RECORD_ID_INPUTS = list(BRONZE_CFG["record_id_inputs"])
RECORD_ID_METHOD = BRONZE_CFG["record_id_method"]

# DataFrame-friendly
LABEL_TYPE_DF = pd.NA if LABEL_TYPE is None else LABEL_TYPE
LABEL_SOURCE_DF = pd.NA if LABEL_SOURCE is None else LABEL_SOURCE



BRONZE_TRAIN_DATA_FILE_NAME = FILENAMES["bronze_train_file_name"]


#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

BRONZE_SOURCE_MODE = "file"   # "file" | "postgres_handoff"


#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

DATASET_NAME_POSTGRES = "pump"
POSTGRES_SOURCE_TABLE_NAME = "bronze_observations_input_stage"

POSTGRES_SOURCE_TABLE_DATASET_MAP = {
    POSTGRES_SOURCE_TABLE_NAME: str(DATASET_NAME_POSTGRES).strip(),
}

----

## Start Logging for the Bronze Stage

Before I begin the actual data work, I want logging turned on so the notebook records what happened during the run.

This is useful for tracing the workflow later, especially when I need to check what file paths were used, whether a step ran successfully, or where something may have gone wrong. In a project like this, logging helps make the notebook feel more like a repeatable pipeline step instead of a one-off analysis.

In [ ]:

# Create bronze log path 
bronze_log_path = paths.logs / "bronze.log"

# Insure directory exists
bronze_log_path.parent.mkdir(parents=True, exist_ok=True)

# Initial Logger
capstone_logger = configure_logging(
    "capstone",
    bronze_log_path,
    level=logging.DEBUG,
    overwrite_handlers=True,
)

# Initiate Logger and log file
logger = logging.getLogger("capstone.bronze")

# Log load and initiation
logger.info("Bronze stage starting")

log_layer_paths(paths, current_layer="bronze", logger=logger)


----

## Define Dataset Name Resolution Logic

This function handles one small but important preprocessing detail: deciding what dataset name should be used during Bronze ingestion.

I want the dataset name to be consistent and machine-friendly because it gets reused in file names, truth records, artifact names, and later pipeline steps. This function helps standardize that process instead of depending on loose manual naming.

Even though this is just one helper function, it supports reproducibility. If I rerun the notebook or use the same logic elsewhere, I want the naming behavior to stay predictable.

In [ ]:
def resolve_dataset_name_for_bronze_pre_ingest(
    *,
    argument_value: Optional[str] = None,
    config_value: Optional[str] = None,
    handoff_dataset_name: Optional[str] = None,
    source_table_name: Optional[str] = None,
    source_table_dataset_map: Optional[dict] = None,
    fallback_value: Optional[str] = None,
    source_path: Optional[str] = None,
) -> Tuple[str, str, str]:
    """
    Resolve dataset name before Bronze ingestion.

    Priority order:
    1. CLI / Argument
    2. Config File
    3. Explicit handoff dataset name
    4. Source table -> dataset mapping
    5. Deterministic file-details-based generated name
    6. Fallback
    """

    def _normalize_dataset_name(dataset_name: str) -> str:
        normalized_value = str(dataset_name).strip().lower()
        normalized_value = normalized_value.replace(" ", "_")
        normalized_value = normalized_value.replace("-", "_")

        cleaned_characters = []
        for character in normalized_value:
            if character.isalnum() or character == "_":
                cleaned_characters.append(character)

        normalized_value = "".join(cleaned_characters)

        while "__" in normalized_value:
            normalized_value = normalized_value.replace("__", "_")

        normalized_value = normalized_value.strip("_")

        if normalized_value == "":
            raise ValueError("Dataset name normalization produced an empty value.")

        return normalized_value

    def _generate_deterministic_dataset_name_from_file_details(
        path_value: Optional[str],
    ) -> Optional[str]:
        if path_value is None or str(path_value).strip() == "":
            return None

        path_object = Path(path_value)

        file_stem_raw = path_object.stem.strip()
        if file_stem_raw == "":
            file_stem_raw = "dataset"

        file_stem_normalized = _normalize_dataset_name(file_stem_raw)

        file_size_bytes = "na"
        modified_timestamp = "na"
        content_fingerprint = "nohash"

        if path_object.exists() and path_object.is_file():
            stat_result = path_object.stat()
            file_size_bytes = str(int(stat_result.st_size))
            modified_timestamp = str(int(stat_result.st_mtime))

            try:
                sample_hasher = hashlib.sha1()

                with open(path_object, "rb") as file_handle:
                    first_chunk = file_handle.read(65536)
                    sample_hasher.update(first_chunk)

                    if stat_result.st_size > 65536:
                        seek_position = max(stat_result.st_size - 65536, 0)
                        file_handle.seek(seek_position)
                        last_chunk = file_handle.read(65536)
                        sample_hasher.update(last_chunk)

                sample_hasher.update(file_size_bytes.encode("utf-8"))
                sample_hasher.update(modified_timestamp.encode("utf-8"))

                content_fingerprint = sample_hasher.hexdigest()[:8]

            except Exception:
                content_fingerprint = "readfail"

        generated_dataset_name = (
            f"{file_stem_normalized}_{file_size_bytes}_{modified_timestamp}_{content_fingerprint}"
        )

        return _normalize_dataset_name(generated_dataset_name)

    source_table_dataset_map = source_table_dataset_map or {}

    if argument_value is not None and str(argument_value).strip() != "":
        return (
            _normalize_dataset_name(str(argument_value)),
            "argument",
            "argument",
        )

    if config_value is not None and str(config_value).strip() != "":
        return (
            _normalize_dataset_name(str(config_value)),
            "config",
            "config",
        )

    if handoff_dataset_name is not None and str(handoff_dataset_name).strip() != "":
        return (
            _normalize_dataset_name(str(handoff_dataset_name)),
            "handoff_dataset_name",
            "handoff",
        )

    if source_table_name is not None and str(source_table_name).strip() != "":
        mapped_dataset_name = source_table_dataset_map.get(str(source_table_name).strip())
        if mapped_dataset_name is not None and str(mapped_dataset_name).strip() != "":
            return (
                _normalize_dataset_name(str(mapped_dataset_name)),
                "source_table_name",
                "source_table_map",
            )

    generated_dataset_name = _generate_deterministic_dataset_name_from_file_details(source_path)

    if generated_dataset_name is not None:
        return (
            generated_dataset_name,
            "source_path",
            "file_details",
        )

    fallback_value_text = (
        fallback_value
        if (fallback_value is not None and str(fallback_value).strip() != "")
        else "unknown_dataset"
    )

    return (
        _normalize_dataset_name(str(fallback_value_text)),
        "fallback",
        "fallback",
    )

## Resolve the Provisional Dataset Identity

Now I am actually calling the dataset naming logic to get an initial dataset identity before the raw file is ingested.

I call this provisional because it is the best dataset name I can determine *before* inspecting the dataset itself. Later in the workflow, the ingest step may confirm that name or improve it if the file contains stronger evidence about what the dataset should be called.

In [ ]:


PROVISIONAL_DATASET_NAME, PROVISIONAL_DATASET_SOURCE_COLUMN, PROVISIONAL_DATASET_METHOD = (
    resolve_dataset_name_for_bronze_pre_ingest(
        argument_value=DATASET_NAME_ARGUMENT,
        config_value=DATASET_NAME_CONFIG,
        handoff_dataset_name=(
            DATASET_NAME_POSTGRES if BRONZE_SOURCE_MODE == "postgres_handoff" else None
        ),
        source_table_name=(
            POSTGRES_SOURCE_TABLE_NAME if BRONZE_SOURCE_MODE == "postgres_handoff" else None
        ),
        source_table_dataset_map=POSTGRES_SOURCE_TABLE_DATASET_MAP,
        fallback_value="unknown_dataset",
        source_path=(
            str(RAW_FILE_PATH / RAW_FILE_NAME) if BRONZE_SOURCE_MODE == "file" else None
        ),
    )
)

In [ ]:
def write_dataset_resolution_attrs(
    dataframe,
    *,
    dataset_column: str = "meta__dataset",
    fallback_dataset_name: str | None = None,
    fallback_method: str = "fallback_dataset_name",
):
    """
    Write Bronze dataset resolution metadata into dataframe.attrs.

    Resolution order:
    1. Use dataset_column if it exists and contains exactly one non-null value
    2. Otherwise use fallback_dataset_name
    """

    resolved_dataset_name = None
    dataset_source_column = None
    dataset_method = None

    if dataset_column in dataframe.columns:
        dataset_values = (
            dataframe[dataset_column]
            .dropna()
            .astype(str)
            .str.strip()
        )

        unique_dataset_values = sorted(value for value in dataset_values.unique() if value)

        if len(unique_dataset_values) == 1:
            resolved_dataset_name = unique_dataset_values[0]
            dataset_source_column = dataset_column
            dataset_method = "dataset_column"

        elif len(unique_dataset_values) > 1:
            raise ValueError(
                f"Multiple dataset values found in '{dataset_column}': {unique_dataset_values}"
            )

    if resolved_dataset_name is None:
        if fallback_dataset_name is None or str(fallback_dataset_name).strip() == "":
            raise ValueError(
                "Could not resolve dataset name from dataframe column or fallback_dataset_name."
            )

        resolved_dataset_name = str(fallback_dataset_name).strip()
        dataset_source_column = None
        dataset_method = fallback_method

    dataframe.attrs["dataset_resolution"] = {
        "dataset_name": resolved_dataset_name,
        "dataset_source_column": dataset_source_column,
        "dataset_method": dataset_method,
    }

    return dataframe

## Save a Snapshot of the Resolved Configuration

Here I am exporting a config snapshot for this run.

This is one of those steps that helps with repeatability and project tracking. By saving the resolved config, I have a record of what settings were actually used for this notebook run, even if the main config files get updated later.

That matters for a Capstone project because I want to be able to connect outputs back to the exact setup that produced them.

----

In [ ]:
CONFIG_SNAPSHOT_PATH = (
    BRONZE_ARTIFACTS_PATH / f"{PROVISIONAL_DATASET_NAME}__bronze__resolved_config.yaml"
)

if CONFIG["execution"].get("save_config_snapshot", True):
    export_config_snapshot(CONFIG, CONFIG_SNAPSHOT_PATH)

----

## Initialize the Experiment Tracking Run

This cell starts the Weights & Biases run for the Bronze stage.

The purpose here is to track key run details such as the dataset identity, version, source file, run metadata, and output location. I am not using W&B to change the data itself. I am using it to document the run so the preprocessing stage is easier to audit and compare later.

In other words, this is part of the project recordkeeping side of the notebook.

In [ ]:
if BRONZE_SOURCE_MODE == "file":
    WANDB_SOURCE_KIND = "file"
    WANDB_SOURCE_REFERENCE = str(RAW_FILE_PATH / RAW_FILE_NAME)
    WANDB_SOURCE_TABLE_NAME = None
    WANDB_RAW_DATA_FILE = RAW_FILE_NAME
    WANDB_RAW_PATH = str(RAW_FILE_PATH / RAW_FILE_NAME)

elif BRONZE_SOURCE_MODE == "postgres_handoff":
    WANDB_SOURCE_KIND = "postgres_handoff"
    WANDB_SOURCE_REFERENCE = f"postgres:public.{POSTGRES_SOURCE_TABLE_NAME}"
    WANDB_SOURCE_TABLE_NAME = POSTGRES_SOURCE_TABLE_NAME
    WANDB_RAW_DATA_FILE = None
    WANDB_RAW_PATH = None

else:
    raise ValueError(f"Unsupported BRONZE_SOURCE_MODE for W&B setup: {BRONZE_SOURCE_MODE}")

In [ ]:
wandb_run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=WANDB_RUN_NAME,
    job_type="bronze",
    config={
        "dataset_name_provisional": PROVISIONAL_DATASET_NAME,
        "dataset_resolution_stage": "pre_ingest",
        "bronze_version": BRONZE_VERSION,

        "source_mode": BRONZE_SOURCE_MODE,
        "source_kind": WANDB_SOURCE_KIND,
        "source_reference": WANDB_SOURCE_REFERENCE,
        "source_table_name": WANDB_SOURCE_TABLE_NAME,

        "raw_data_file": WANDB_RAW_DATA_FILE,
        "raw_path": WANDB_RAW_PATH,

        "split_status": SPLIT_STATUS,
        #"label_type": LABEL_TYPE,
        #"label_source": LABEL_SOURCE,
        "run_id": RUN_ID,
        "asset_id": ASSET_ID,

        "add_record_id": ADD_RECORD_ID,
        "record_id_inputs": RECORD_ID_INPUTS if ADD_RECORD_ID else None,
        "record_id_method": RECORD_ID_METHOD if ADD_RECORD_ID else None,

        "bronze_out_path": str(BRONZE_DATA_PATH),
    },
)

logger.info("W&B initialized: %s", wandb_run.name)

----

## Ingest the Raw Dataset into the Bronze Layer

This is the point where the notebook actually loads the raw data into a DataFrame.

The Bronze layer is focused on getting the raw dataset into a structured, traceable, and standardized form. So in this step I am bringing the file in, attaching key metadata, validating the structure, and preparing it to become the foundation for the rest of the pipeline.

This is the first true data-processing step in the notebook.

In [ ]:
if BRONZE_SOURCE_MODE == "file":
    dataframe = ingest_data(
        RAW_FILE_PATH,
        file_name=RAW_FILE_NAME,
        dataset_name=DATASET_NAME_ARGUMENT,
        dataset_name_config=DATASET_NAME_CONFIG,
        dataset_candidates=DATASET_CANDIDATES,
        split=SPLIT_STATUS,
        run_id=RUN_ID,
        asset_id=ASSET_ID,
        add_record_id=True,
        validate=True,
    )

elif BRONZE_SOURCE_MODE == "postgres_handoff":
    engine = get_engine_from_env()

    dataframe = read_layer_dataframe(
        engine=engine,
        schema="public",
        table_name="bronze_observations_input_stage",
        where_clause="dataset_id = :dataset_id AND run_id = :run_id",
        params={
            "dataset_id": DATASET_NAME_POSTGRES,
            "run_id": RUN_ID,
        },
        order_by="batch_id, row_in_batch",
        require_exists=True,
    )

    dataframe = write_dataset_resolution_attrs(
        dataframe,
        dataset_column="meta__dataset",
        fallback_dataset_name=DATASET_NAME_POSTGRES,
        fallback_method="postgres_handoff",
    )

else:
    raise ValueError(f"Unsupported BRONZE_SOURCE_MODE: {BRONZE_SOURCE_MODE}")

----

## Confirm the Final Dataset Identity After Ingestion

After the ingest step, I want to confirm what dataset identity was actually resolved.

This matters because the ingest process may find stronger evidence inside the dataset itself than what I had available before loading it. So this cell checks the metadata written during ingestion and makes sure the final dataset name and related resolution details are present and usable.

I do this check here so I can trust the dataset identity before continuing into lineage tracking, truth creation, and output saving.

In [ ]:
dataset_resolution = dataframe.attrs.get("dataset_resolution", {})

if not dataset_resolution:
    raise ValueError(
        "Bronze ingest did not write dataset_resolution metadata to dataframe.attrs."
    )

RESOLVED_DATASET_NAME = dataset_resolution.get("dataset_name")
DATASET_SOURCE_COLUMN = dataset_resolution.get("dataset_source_column")
DATASET_METHOD = dataset_resolution.get("dataset_method")

if RESOLVED_DATASET_NAME is None:
    raise ValueError("Bronze ingest did not return dataset_resolution metadata in dataframe.attrs.")

----

## Update Tracking Metadata with the Final Dataset Name

Now that the dataset identity has been confirmed, I update the experiment tracking run so it reflects the final resolved values instead of only the provisional ones.

This keeps the run metadata aligned with the actual ingested dataset and avoids confusion later when I look back at artifacts, logs, or outputs tied to this Bronze execution.

In [ ]:
wandb_run.config.update(
    {
        "dataset_name_final": RESOLVED_DATASET_NAME,
        "dataset_source_column": DATASET_SOURCE_COLUMN,
        "dataset_method": DATASET_METHOD,
        "dataset_resolution_stage": "bronze_ingest_final",
        "source_mode": BRONZE_SOURCE_MODE,
        "source_kind": WANDB_SOURCE_KIND,
        "source_reference": WANDB_SOURCE_REFERENCE,
        "source_table_name": WANDB_SOURCE_TABLE_NAME,
    },
    allow_val_change=True,
)

---

## Log Any Dataset Name Changes

This is a small checkpoint to capture whether the dataset name changed between the pre-ingest estimate and the post-ingest final result.

I like keeping this kind of note in the workflow because it makes the naming process transparent. If the dataset identity changes here, that is not necessarily a problem. It just means the ingest step found better evidence and the notebook adjusted accordingly.

In [ ]:
if PROVISIONAL_DATASET_NAME != RESOLVED_DATASET_NAME:
    logger.info(
        "Dataset name changed after Bronze ingest when inside-dataset evidence became available | provisional=%s | resolved=%s",
        PROVISIONAL_DATASET_NAME,
        RESOLVED_DATASET_NAME,
    )

DATASET_NAME = RESOLVED_DATASET_NAME

----

## Build the Bronze Truth Record Foundation

Here I am creating the base truth record for the Bronze layer.

The truth record is part of the lineage and audit structure for this project. It helps answer questions like:
- what dataset was used
- what version of the pipeline ran
- what file was ingested
- what run metadata applies to this output
- where the resulting artifacts were saved

This is important because I do not want Bronze outputs to exist without context. I want each major output to be tied back to a traceable processing record.

In [ ]:
bronze_truth = initialize_layer_truth(
    truth_version=TRUTH_VERSION,
    dataset_name=DATASET_NAME,
    layer_name=LAYER_NAME,
    process_run_id=PROCESS_RUN_ID,
    pipeline_mode=PIPELINE_MODE,
    parent_truth_hash=None,
)

bronze_truth = update_truth_section(
    bronze_truth,
    "config_snapshot",
    {
        "bronze_version": BRONZE_VERSION,
        "split_status": SPLIT_STATUS,
        "label_type": LABEL_TYPE,
        "label_source": LABEL_SOURCE,
        "run_id": RUN_ID,
        "asset_id": ASSET_ID,
        "add_record_id": ADD_RECORD_ID,
        "record_id_inputs": RECORD_ID_INPUTS if ADD_RECORD_ID else None,
        "record_id_method": RECORD_ID_METHOD if ADD_RECORD_ID else None,
        "pipeline_mode": PIPELINE_MODE,
    },
)

bronze_truth = update_truth_section(
    bronze_truth,
    "runtime_facts",
    {
        "source_run_id": RUN_ID,
        "raw_file_path": str(RAW_FILE_PATH / RAW_FILE_NAME),
        "raw_data_dir": str(RAW_DATA_PATH),
        "dataset_name_provisional": PROVISIONAL_DATASET_NAME,
        "dataset_name_final": RESOLVED_DATASET_NAME,
        "dataset_source_column": DATASET_SOURCE_COLUMN,
        "dataset_method": DATASET_METHOD,
    },
)

bronze_truth = update_truth_section(
    bronze_truth,
    "artifact_paths",
    {
        "bronze_output_dir": str(BRONZE_DATA_PATH),
        "bronze_output_file_name": "pump__bronze__train.parquet",
    },
)

bronze_truth = update_truth_section(
    bronze_truth,
    "notes",
    {
        "purpose": "Bronze ingestion truth record",
    },
)

----

## Bronze Data Review

Before moving forward, I want to stop and look at what the Bronze dataframe actually looks like.

At this stage, I am not doing deep feature engineering or final analysis yet. What I *am* doing is checking the overall structure of the ingested data so I can confirm that the Bronze layer did what I expected it to do.

I want to review:
- the shape of the dataframe
- the data types
- memory usage
- a sample of the rows
- basic numeric and categorical summaries
- which columns are metadata columns
- whether there is any obvious missingness showing up already

This kind of review gives me a practical first look at the dataset before I lock in saves, profiling outputs, and lineage records.

In [ ]:
# Basic Dataframe Information/Summary

# Shape 
print("Shape:", dataframe.shape)
logger.info("Shape: %s", dataframe.shape)

# Dtypes as a compact block
print("\nData types:")
print(dataframe.dtypes)
logger.info("Dtypes:\n%s", dataframe.dtypes.to_string())

# Memory Usages
print("\nMemory usage (MB):")
print(dataframe.memory_usage(deep=True).sum() / (1024 ** 2))
mem_mb = dataframe.memory_usage(deep=True).sum() / (1024 ** 2)
logger.info("Memory usage (MB): %.2f", mem_mb)

# Head(15) as text (truncate columns for readability)
print("\nFirst 15 rows:")
display(dataframe.head(15))
logger.info("Head(15):\n%s", dataframe.head(15).to_string(max_cols=40, max_rows=15))

# Describe Numbers
print("\nBasic numeric summary:")
display(dataframe.describe(include=[np.number]).T)
desc_num = dataframe.describe(include=[np.number]).T
logger.info("Numeric describe (truncated):\n%s", desc_num.to_string(max_rows=60))

# Describe Objects and categorical
print("\nBasic object / categorical summary:")
object_category_columns = dataframe.select_dtypes(include=["object", "category", "string"]).columns
if len(object_category_columns):
    desc_obj = dataframe[object_category_columns].describe().T
    display(desc_obj)
    logger.info("Object/categorical describe (truncated):\n%s", desc_obj.to_string(max_rows=60))
else:
    logger.info("No object/category/string columns to describe.")

# Meta Columns
print("\nMeta Columns Summary:")
meta_columns = [column for column in dataframe.columns if column.startswith("meta__")]
dataframe[meta_columns].head(3)
logger.info("Meta Columns: (%d): %s", len(meta_columns), "\n".join(meta_columns))

# All Other Columns
print("\nAll Other Columns Summary:")
other_columns = [column for column in dataframe.columns if not column.startswith("meta__")]
dataframe[other_columns].head(3)
logger.info("All Other Columns: (%d): %s", len(other_columns), "\n".join(other_columns))

# Missing
missing = (dataframe[other_columns].isna().mean() * 100).sort_values(ascending=False).head(20)
display(missing.to_frame("pct_missing"))
logger.info("Top missingness (%%):\n%s", missing.to_string())


----

### Ask

What does this Bronze review tell me, and is the dataset in a good enough state to move forward?

### Answer

This check gives me a first-pass structural read on the ingested dataset. I am looking less for deep business insight here and more for confirmation that the Bronze process worked correctly.

A few things matter most at this step:
- the row and column counts should make sense for the raw source
- the data types should look reasonable based on the sensor-style fields and metadata fields
- the `meta__` columns should be present because they support lineage and tracking
- missing values should be noted now, even if I do not address them until later stages
- the sample rows should look clean enough that I trust the ingestion step

So the purpose of this section is not to make modeling decisions yet. It is to answer a more basic question: **did I successfully bring the raw dataset into a structured Bronze form without losing context or breaking the schema?**

---

## Finalize Lineage Metadata and Save the Bronze Truth Record

Now I am moving from general setup into formal lineage tracking.

This step fingerprints the source file, identifies the meta and feature columns, builds the full Bronze truth record, generates the Bronze truth hash, and stamps the dataframe with the lineage columns that connect the data back to that truth record.

This is one of the most important governance-style steps in the notebook because it makes the dataset traceable. Once this is done, the Bronze dataframe is no longer just a loaded table. It becomes a tracked pipeline artifact with documented provenance.

In [ ]:
bronze_source_fingerprint = build_file_fingerprint(RAW_FILE_PATH / RAW_FILE_NAME)
bronze_truth = update_truth_section(
    bronze_truth,
    "source_fingerprint",
    bronze_source_fingerprint,
)

bronze_meta_columns = sorted(
    set(
        identify_meta_columns(dataframe)
        + [
            "meta__truth_hash",
            "meta__parent_truth_hash",
            "meta__pipeline_mode",
        ]
    )
)

bronze_feature_columns = identify_feature_columns(dataframe)

bronze_truth_record = build_truth_record(
    truth_base=bronze_truth,
    row_count=len(dataframe),
    column_count=dataframe.shape[1] + 3,
    meta_columns=bronze_meta_columns,
    feature_columns=bronze_feature_columns,
)

BRONZE_TRUTH_HASH = bronze_truth_record["truth_hash"]

dataframe = stamp_truth_columns(
    dataframe,
    truth_hash=BRONZE_TRUTH_HASH,
    parent_truth_hash=None,
    pipeline_mode=PIPELINE_MODE,
)

bronze_truth_path = save_truth_record(
    bronze_truth_record,
    truth_dir=TRUTHS_PATH,
    dataset_name=DATASET_NAME,
    layer_name=LAYER_NAME,
)

append_truth_index(
    bronze_truth_record,
    truth_index_path=TRUTH_INDEX_PATH,
)

logger.info("Bronze truth hash: %s", BRONZE_TRUTH_HASH)
logger.info("Bronze truth path: %s", bronze_truth_path)
logger.info("Bronze process_run_id: %s", PROCESS_RUN_ID)

print("Bronze truth hash:", BRONZE_TRUTH_HASH)
print("Bronze truth path:", bronze_truth_path)
print("Bronze process_run_id:", PROCESS_RUN_ID)

----

## Define a Helper to Reorder Bronze Columns

This helper function keeps the dataframe layout consistent by moving the metadata columns to the front.

I do this mostly for clarity and standardization. The data would still exist either way, but putting the `meta__` columns first makes the Bronze output easier to inspect, easier to debug, and more consistent with the rest of the pipeline structure.

In [ ]:
def collect_meta_columns(existing_columns: List[str]) -> List[str]:
    meta_columns: List[str] = []
    for column in existing_columns:
        if column.startswith("meta__"):
            meta_columns.append(column)
    return meta_columns



def reorder_bronze_columns(dataframe: pd.DataFrame) -> pd.DataFrame:
    existing_columns = list(dataframe.columns)

    meta_columns = collect_meta_columns(existing_columns)

    bronze_columns: List[str] = []
    for column in existing_columns:
        if column not in meta_columns:
            bronze_columns.append(column)

    final_order: List[str] = []
    final_order.extend(meta_columns)
    final_order.extend(bronze_columns)

    return dataframe[final_order].copy()

## Reorder the Bronze Dataframe Columns

Here I apply the helper function so the metadata columns come first and the feature columns follow after them.

This does not change the values in the dataset. It only changes the column order. I consider this a cleanup and readability step so the saved Bronze output is easier to work with downstream.

In [ ]:
# Reorder dataframe and put meta columns in the front. 

dataframe = reorder_bronze_columns(dataframe)

meta_columns_after_reorder = collect_meta_columns(list(dataframe.columns))

non_meta_columns_after_reorder = [
    column_name
    for column_name in dataframe.columns
    if not column_name.startswith("meta__")
]

logger.info(
    "Bronze columns reordered successfully. "
    "Meta columns moved to the front while preserving original within-group order. "
    "meta_column_count=%d | non_meta_column_count=%d | total_column_count=%d",
    len(meta_columns_after_reorder),
    len(non_meta_columns_after_reorder),
    dataframe.shape[1],
)


----

## Save the Bronze Dataset

At this point the Bronze dataframe has been ingested, reviewed, stamped with lineage information, and reordered into a cleaner structure.

Now I save it as the Bronze layer output so it can be reused by later notebooks and processing stages without having to repeat all of the earlier steps.

In [ ]:
# Save Data as Parquet
save_data(dataframe, paths.data_bronze_train, BRONZE_TRAIN_DATA_FILE_NAME)

----

## Create Bronze Profiling Outputs

This step generates profiling and diagnostic outputs for the Bronze dataframe.

The main reason I do this is to preserve a record of the dataset's condition at this stage. Instead of relying only on what I saw in the notebook during one run, I also save profiling results as artifacts that can be reviewed later.

That makes it easier to compare stages and to document how the data looked before later cleaning and transformation steps.

In [ ]:
metrics, saved = profile_dataframe(dataframe, logger, artifacts_dir=BRONZE_ARTIFACTS_PATH)

----

## Finalize Bronze Run Tracking

Now I am wrapping the Bronze stage into a cleaner tracked output by sending the final dataset, logs, and notebook references into the run finalization step.

This helps tie together the data product, the experiment tracking record, and the supporting files from this stage. It is part of treating the notebook like a real pipeline step rather than only an interactive worksheet.

In [ ]:
finalize_wandb_stage(
    wandb_run,
    stage="bronze",
    dataframe=dataframe,
    project_root=paths.root,
    logs_dir=paths.logs,
    dataset_dirs=[paths.data_bronze_train],
    dataset_artifact_name="pump__bronze__train.parquet",
    notebook_path=paths.notebooks / "Preprocessing"/ "EDA_Notebook_Bronze_01_Preprocessing.ipynb",
)

----

In [ ]:
wandb_run.finish()

----

In [ ]:
'''

# Ensure artifacts subdir exists
bronze_art_dir = paths.artifacts / "bronze"
bronze_art_dir.mkdir(parents=True, exist_ok=True)

# Profile + export describe() CSVs into artifacts/bronze
metrics, saved = profile_dataframe(dataframe, logger, artifacts_dir=bronze_art_dir, head=15)

# Start or reuse a W&B run (you can also init at the top; either is fine)
run = wandb.init(project="capstone", job_type="bronze", config={"dataset": "pump", "stage": "bronze"})

# Log scalars + preview table near the end
log_metrics(run, metrics)
log_dataframe_head(run, dataframe, key="bronze_head15", n=15)

# Upload logs (just bronze.log)
log_files_as_artifact(
    run,
    artifact_name="capstone-logs-bronze",
    artifact_type="logs",
    files=[paths.logs / "bronze.log"],
    aliases=["latest"],
    metadata={"stage": "bronze", "dataset": "pump"},
)

# Upload bronze parquet outputs (train dir)
log_dir_as_artifact(
    run,
    artifact_name="pump-bronze-train-parquet",
    artifact_type="dataset",
    dir_path=paths.data_bronze_train,
    patterns=("*.parquet", "*.pq"),  # explicit
    aliases=["latest"],
    metadata={"stage": "bronze", "split": "train", "dataset": "pump"},
)

# Upload bronze diagnostics exports (artifacts/bronze)
log_dir_as_artifact(
    run,
    artifact_name="pump-bronze-diagnostics",
    artifact_type="eda",
    dir_path=bronze_art_dir,
    patterns=("*.csv", "*.json", "*.log", "*.txt"),
    aliases=["latest"],
    metadata={"stage": "bronze", "dataset": "pump"},
)

# Upload the notebook itself (point this at where you actually save it)
# Best practice: save/copy the notebook into paths.notebooks first.
bronze_nb = paths.notebooks / "Bronze_Preprocessing_Pump.ipynb"

if bronze_nb.exists():
    log_files_as_artifact(
        run,
        artifact_name="capstone-notebooks",
        artifact_type="notebook",
        files=[bronze_nb],
        aliases=["latest"],
        metadata={"stage": "bronze"},
    )
else:
    logger.warning("Notebook not found at %s; skipping upload.", bronze_nb)

run.finish()

'''


----

## Validate Bronze Lineage and Truth Consistency

Before I consider the Bronze stage complete, I want to run a sanity check on the lineage fields and saved truth record.

This is basically a trust check. I want to make sure:
- the required lineage columns exist
- the dataframe truth hash matches the saved truth record
- Bronze does not incorrectly claim a parent truth hash
- the saved truth file exists
- the recorded row and column counts match the actual dataframe

This step gives me confidence that the Bronze output is not only saved, but also internally consistent.

In [ ]:
required_bronze_meta_columns = [
    "meta__truth_hash",
    "meta__parent_truth_hash",
    "meta__pipeline_mode",
]

missing_bronze_meta_columns = [
    column_name
    for column_name in required_bronze_meta_columns
    if column_name not in dataframe.columns
]
if missing_bronze_meta_columns:
    raise ValueError(
        f"Bronze dataframe is missing required lineage columns: {missing_bronze_meta_columns}"
    )

bronze_dataframe_truth_hash = extract_truth_hash(dataframe)
if bronze_dataframe_truth_hash is None:
    raise ValueError("Bronze dataframe does not contain a readable meta__truth_hash value.")

if bronze_dataframe_truth_hash != BRONZE_TRUTH_HASH:
    raise ValueError(
        "Bronze dataframe truth hash does not match BRONZE_TRUTH_HASH:\n"
        f"dataframe={bronze_dataframe_truth_hash}\n"
        f"record={BRONZE_TRUTH_HASH}"
    )

if "meta__parent_truth_hash" in dataframe.columns:
    bronze_parent_values = dataframe["meta__parent_truth_hash"].dropna().astype(str).unique().tolist()
    if bronze_parent_values:
        raise ValueError(
            "Bronze should not have a populated parent truth hash, but found values:\n"
            f"{bronze_parent_values}"
        )

if not Path(bronze_truth_path).exists():
    raise FileNotFoundError(f"Bronze truth file was not created: {bronze_truth_path}")

loaded_bronze_truth = load_json(bronze_truth_path)

if loaded_bronze_truth.get("truth_hash") != BRONZE_TRUTH_HASH:
    raise ValueError(
        "Saved Bronze truth file hash does not match BRONZE_TRUTH_HASH:\n"
        f"file={loaded_bronze_truth.get('truth_hash')}\n"
        f"record={BRONZE_TRUTH_HASH}"
    )

if loaded_bronze_truth.get("parent_truth_hash") is not None:
    raise ValueError(
        "Bronze truth file parent_truth_hash should be None.\n"
        f"Found: {loaded_bronze_truth.get('parent_truth_hash')}"
    )

if loaded_bronze_truth.get("row_count") != len(dataframe):
    raise ValueError(
        "Bronze truth row_count does not match dataframe row count:\n"
        f"truth={loaded_bronze_truth.get('row_count')}\n"
        f"dataframe={len(dataframe)}"
    )

if loaded_bronze_truth.get("column_count") != dataframe.shape[1]:
    raise ValueError(
        "Bronze truth column_count does not match stamped dataframe column count:\n"
        f"truth={loaded_bronze_truth.get('column_count')}\n"
        f"dataframe={dataframe.shape[1]}"
    )

print("Bronze lineage sanity check passed.")

----

## Final Bronze Structural Check

I am doing one more quick review of the dataframe before moving on from the Bronze notebook.

This is another lightweight inspection step. The earlier review was more detailed, while this one acts more like a final confirmation that the dataset still looks correct after lineage stamping, column reordering, and saving steps.

In [ ]:
print("=== Silver Pre-EDA: Data Overview ===")
print(f"Shape: {dataframe.shape[0]:,} rows × {dataframe.shape[1]} columns\n")

print("=== Column Types ===")
display(dataframe.dtypes.to_frame("dtype").head(20))

print("\n=== df.info() ===")
dataframe.info()

print("\n=== Basic Descriptive Statistics (numeric only) ===")
display(dataframe.describe().T)

print("\n=== Get Top Sample Rows")
display(dataframe.head())

### Ask

What does this final review confirm about the Bronze output?

### Answer

This final check helps confirm that the Bronze dataframe still looks structurally sound after all preprocessing steps have been applied.

At this point, I want to see that:
- the dataframe still has the expected dimensions
- the columns and dtypes still look reasonable
- the descriptive statistics do not show anything obviously broken
- the top sample rows still match the kind of data I expect to see in Bronze

This is less about deep analysis and more about confirming readiness. In other words, the question here is: **does this Bronze output look stable enough to hand off to the next stage of the pipeline?**

If the answer is yes, then this notebook has done its job. The raw source has been turned into a standardized, traceable Bronze dataset that is ready for downstream work.

----

## Optional Database Write Step

This section is for writing the Bronze dataframe into PostgreSQL.

I am treating this as an optional persistence step rather than a required part of the core Bronze preprocessing flow. The main Bronze deliverable is still the saved dataset and its supporting lineage artifacts. Writing to SQL is useful when I want the Bronze layer available in a database for later querying, validation, or downstream integration work.

SQL_SCHEMAS = {
    "bronze": "bronze",
    "silver": "silver",
    "gold": "gold",
    "synthetic": "synthetic",
    "truth": "truth",
    "audit": "audit",
}

GOLD_ARTIFACT_TABLES = {
    "baseline_results": "baseline_results",
    "baseline_summary": "baseline_summary",
    "baseline_thresholds": "baseline_thresholds",
    "cascade_results": "cascade_results",
    "cascade_summary": "cascade_summary",
    "cascade_thresholds": "cascade_thresholds",
    "comparison_summary": "comparison_summary",
}

SYNTHETIC_ARTIFACT_TABLES = {
    "batch": "batch",
    "stream": "stream",
    "sensor_messages": "sensor_messages",
}


engine = get_engine_from_env()


BRONZE_SCHEMA = SQL_SCHEMAS["bronze"]

bronze_sql_dataframe = prepare_layer_dataframe(
    dataframe,
    truth_hash=bronze_truth_record,
    parent_truth_hash=bronze_parent_values,
    pipeline_mode=PIPELINE_MODE,
    process_run_id=PROCESS_RUN_ID,
    add_loaded_at_column=True,
)

bronze_table_name = write_layer_dataframe(
    engine=engine,
    dataframe=bronze_sql_dataframe,
    schema=BRONZE_SCHEMA,
    dataset_name=DATASET_NAME,
    if_exists="replace",   # use "append" when appropriate
    index=False,
)

print(f"Wrote table: {BRONZE_SCHEMA}.{bronze_table_name}")